In [1]:
import os
import sys
import numpy as np
from pathlib import Path

# Add src to path (notebook is in src/models/, need to go up 2 levels to workspace root)
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from src.config import PROCESSED_DIR, GENERATED_MIDIS_DIR, START_PITCH
from src.models.baselines import RandomNoteGenerator, MarkovChainMusic

print("Loading training data for Markov Chain...")
train_data_path = PROCESSED_DIR / "maestro_train_windows.npy"

if train_data_path.exists():
    train_windows = np.load(train_data_path)
    print(f"Loaded {len(train_windows)} windows.")
    
    # We need to extract simple sequences of pitches for the Markov Chain.
    # We will grab the highest active pitch at each timestep to form a melody sequence.
    print("Extracting melodic sequences...")
    sequences = []
    for window in train_windows[:5000]: # Limit to 5000 windows for speed
        seq = []
        for time_step in window:
            active_pitches = np.where(time_step == 1)[0]
            if len(active_pitches) > 0:
                # Add START_PITCH to convert back to standard MIDI note numbers
                highest_pitch = active_pitches[-1] + START_PITCH
                seq.append(highest_pitch)
        if len(seq) > 1:
            sequences.append(seq)
            
    # 1. Train and Generate Markov Baseline
    print("Fitting Markov Chain...")
    markov_model = MarkovChainMusic()
    markov_model.fit(sequences)
    
    markov_path = GENERATED_MIDIS_DIR / "markov_baseline.mid"
    markov_model.generate(start_note=60, num_notes=100, output_path=markov_path)
    print(f"Saved Markov Chain baseline to {markov_path}")

else:
    print(f"Train data not found at {train_data_path}. Please run Preprocessing first.")

# 2. Generate Random Baseline
print("Generating Random Note baseline...")
random_model = RandomNoteGenerator()
random_path = GENERATED_MIDIS_DIR / "random_baseline.mid"
random_model.generate(num_notes=100, output_path=random_path)
print(f"Saved Random baseline to {random_path}")

print("\nBaselines complete! You should listen to the generated files in outputs/generated_midis/")

Loading training data for Markov Chain...
Loaded 57441 windows.
Extracting melodic sequences...
Fitting Markov Chain...
Saved Markov Chain baseline to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\markov_baseline.mid
Generating Random Note baseline...
Saved Random baseline to Y:\.college\CSE425\.project\music_generation_unsupervised\outputs\generated_midis\random_baseline.mid

Baselines complete! You should listen to the generated files in outputs/generated_midis/
